# Signalement des anomalies d'adresses email — EG FINESS

## 1. Imports et connexion

In [1]:
%run ../../config/config.ipynb

import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path

from src.email_checker import (
    analyser_emails, marquer_doublons, evaluer_correspondance,
    charger_tlds_iana, charger_bases_noms,
)

pd.set_option("display.max_colwidth", 80)
print("Imports OK")

Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Note: you may need to restart the kernel to use updated packages.
Connexion OK
Imports OK


## 2. Référentiels externes — TLD IANA + bases de noms (filtrées par fréquence) + base communes france

In [2]:
DOSSIER_REF = Path("../../data/referentiels")

charger_tlds_iana(cache=DOSSIER_REF / "tlds-alpha-by-domain.txt")

df_prenoms    = pd.read_csv(DOSSIER_REF / "prenom.csv",     sep=",", dtype=str)
df_patronymes = pd.read_csv(DOSSIER_REF / "patronymes.csv", sep=",", dtype=str)

charger_bases_noms(df_prenoms, df_patronymes)

from src.email_checker import PRENOMS, PATRONYMES, TLDS_VALIDES, SEUIL_PRENOM, SEUIL_PATRONYME
print(f"TLD valides                      : {len(TLDS_VALIDES):,}")
print(f"Prénoms retenus (sum >= {SEUIL_PRENOM})     : {len(PRENOMS):,}")
print(f"Patronymes retenus (count >= {SEUIL_PATRONYME}) : {len(PATRONYMES):,}")

TLD valides                      : 1,437
Prénoms retenus (sum >= 500)     : 1,309
Patronymes retenus (count >= 100) : 14,710


In [3]:
df_communes = pd.read_csv(DOSSIER_REF / "communes-france-2026.csv", dtype=str)

from src.email_checker import charger_geo_insee
stats_geo = charger_geo_insee(df_communes, seuil_population=2000)
print(f"Villes retenues (pop >= 2000) : {stats_geo['villes']:,}")
print(f"Départements : {stats_geo['departements']} · Régions : {stats_geo['regions']}")

Villes retenues (pop >= 2000) : 5,200
Départements : 101 · Régions : 18


## 3. Chargement des EG FINESS

In [4]:
query_eg = """
    SELECT
        idstructure_stru,
        nmfinessej_stru,
        nmfinessetab_stru,
        raisonsociale_stru,
        cdcommune_stru,
        lbvoie_stru,
        email_stru
    FROM BICOEUR_DWH_SNAPSHOT.dbo.dwh_structure
    WHERE topsource_stru = 'FINESS'
      AND typeidpm_stru  = 'EG'
      AND (dtfermestruct_stru IS NULL OR dtfermestruct_stru >= SYSDATETIME())
"""

df_eg = pd.read_sql(query_eg, conn)

df_eg["departement"] = df_eg["cdcommune_stru"].astype(str).str[:2]

print(f"EG FINESS actifs chargés : {len(df_eg):,}")

EG FINESS actifs chargés : 104,805


## 4. Vue d'ensemble

In [5]:
total      = len(df_eg)
avec_email = df_eg["email_stru"].notna().sum()
sans_email = total - avec_email
uniques    = df_eg["email_stru"].nunique()

print("─" * 50)
print("PANORAMA DES EMAILS EG FINESS")
print("─" * 50)
print(f"  EG actifs total             : {total:>10,}")
print(f"  Email renseigné             : {avec_email:>10,}  ({avec_email/total*100:.1f} %)")
print(f"  Email absent                : {sans_email:>10,}  ({sans_email/total*100:.1f} %)")
print(f"  Emails uniques              : {uniques:>10,}")
print(f"  Doublons potentiels         : {avec_email - uniques:>10,}")
print("─" * 50)

──────────────────────────────────────────────────
PANORAMA DES EMAILS EG FINESS
──────────────────────────────────────────────────
  EG actifs total             :    104,805
  Email renseigné             :     71,570  (68.3 %)
  Email absent                :     33,235  (31.7 %)
  Emails uniques              :     55,796
  Doublons potentiels         :     15,774
──────────────────────────────────────────────────


## 5. Détection des anomalies

In [6]:
print("Analyse des anomalies + classification...")
df = analyser_emails(df_eg, col_email="email_stru")
df = marquer_doublons(df, col_email_norm="email_norm", col_id="idstructure_stru")
print(f"Analyse terminée — {len(df):,} EG traités.")

Analyse des anomalies + classification...
Analyse terminée — 104,805 EG traités.


## 6. Correspondance partie locale ↔ raison sociale / adresse

In [7]:
query_ej = """
    SELECT
        nmfinessej_stru AS cle_ej,
        raisonsociale_stru AS raison_ej
    FROM BICOEUR_DWH_SNAPSHOT.dbo.dwh_structure
    WHERE topsource_stru = 'FINESS'
      AND typeidpm_stru  = 'EJ'
"""
df_ej_ref = pd.read_sql(query_ej, conn)
map_raison_ej = dict(zip(df_ej_ref["cle_ej"].astype(str),
                         df_ej_ref["raison_ej"].astype(str)))
print(f"Raisons sociales EJ chargées : {len(map_raison_ej):,}")

def _corresp_ligne(row):
    raison_ej = map_raison_ej.get(str(row["nmfinessej_stru"]), "")
    return evaluer_correspondance(
        local_part=row["local_part"],
        raison_principale=row["raisonsociale_stru"],
        raison_parent=raison_ej,
        adresse=row.get("lbvoie_stru", ""),
        libelle_principal="(EG)",
        libelle_parent="(EJ parent)",
    )

df["correspondance"] = ""
masque = df["local_part"] != ""
df.loc[masque, "correspondance"] = df.loc[masque].apply(_corresp_ligne, axis=1)

print("\nRépartition des correspondances :")
print(df[df["correspondance"] != ""]["correspondance"].value_counts().to_string())

Raisons sociales EJ chargées : 98,158

Répartition des correspondances :
correspondance
Aucune correspondance                   39418
Raison sociale (EG)                     27524
Raison sociale (EJ parent)               2490
Raison sociale (EG) + Adresse            1828
Adresse                                   269
Raison sociale (EJ parent) + Adresse       41


In [ ]:
from src.email_checker import detecter_geo

geo = df.apply(lambda row: detecter_geo(row["local_part"], row.get("cdcommune_stru")), axis=1)
df["geo_local"]      = geo.apply(lambda x: x["geo_local"])
df["geo_concordant"] = geo.apply(lambda x: x["geo_concordant"])

apercu = df[(df["geo_local"] != "") & (df["geo_concordant"] == False)]
print(f"Emails portant un lieu : {(df['geo_local'] != '').sum():,}")
print(f"  dont concordants     : {(df['geo_concordant'] == True).sum():,}")
print(f"  dont NON concordants : {len(apercu):,}")
apercu[["raisonsociale_stru", "email_stru", "geo_local"]].head(10)

## 7. Distribution globale des signalements

In [9]:
dist_niveau = df["niveau_anomalie"].value_counts().sort_index()
labels = {0: "Valides", 1: "Critiques (niveau 1)", 2: "Qualité (niveau 2)"}

print("─" * 55)
print("DISTRIBUTION GLOBALE DES SIGNALEMENTS")
print("─" * 55)
for niveau, n in dist_niveau.items():
    print(f"  {labels.get(niveau, f'Niveau {niveau}'):<30} : {n:>8,}  ({n/len(df)*100:5.1f} %)")
print("─" * 55)

print("\nDÉTAIL PAR TYPE D'ANOMALIE")
print("─" * 55)
detail = (
    df[df["niveau_anomalie"] > 0]
    .groupby(["niveau_anomalie", "code_anomalie"])
    .size().reset_index(name="nb")
    .sort_values(["niveau_anomalie", "nb"], ascending=[True, False])
)
for _, r in detail.iterrows():
    print(f"  [{r['niveau_anomalie']}] {r['code_anomalie']:<22} : {r['nb']:>6,}")

───────────────────────────────────────────────────────
DISTRIBUTION GLOBALE DES SIGNALEMENTS
───────────────────────────────────────────────────────
  Valides                        :   30,745  ( 29.3 %)
  Critiques (niveau 1)           :   33,337  ( 31.8 %)
  Qualité (niveau 2)             :   40,723  ( 38.9 %)
───────────────────────────────────────────────────────

DÉTAIL PAR TYPE D'ANOMALIE
───────────────────────────────────────────────────────
  [1] EMAIL_VIDE             : 33,235
  [1] DOMAINE_SANS_POINT     :     33
  [1] TLD_INEXISTANT         :     20
  [1] EXTENSION_VIDE         :     18
  [1] DOMAINE_INEXISTANT     :     16
  [1] EXTENSION_TYPO         :     10
  [1] EXTENSION_LIEU         :      2
  [1] ESPACE                 :      1
  [1] EXTENSION_FICHIER      :      1
  [1] POINTS_CONSECUTIFS     :      1
  [2] DOUBLON                : 20,901
  [2] EMAIL_GRAND_PUBLIC     : 19,822


## 8. Classification de la partie locale

In [10]:
classif = df[df["type_local"] != ""]
print(f"Classification de {len(classif):,} emails\n")

print("Partie locale :")
print(classif["type_local"].value_counts().to_string())
print("\nDomaine :")
print(classif["type_domaine"].value_counts().to_string())
print("\nCroisement local x domaine :")
print(pd.crosstab(classif["type_local"], classif["type_domaine"]).to_string())

Classification de 71,464 emails

Partie locale :
type_local
INDETERMINE          24451
GENERIQUE            21140
INSTITUTIONNEL       16197
NOMINATIF             6700
NOMINATIF_PARTIEL     2976

Domaine :
type_domaine
PROPRE    51642
PUBLIC    19822

Croisement local x domaine :
type_domaine       PROPRE  PUBLIC
type_local                       
GENERIQUE           19485    1655
INDETERMINE         15978    8473
INSTITUTIONNEL       8491    7706
NOMINATIF            5265    1435
NOMINATIF_PARTIEL    2423     553


## 9. Anomalies critiques (niveau 1)

In [ ]:
df_n1 = df[df["niveau_anomalie"] == 1].copy()
print(f"Total anomalies critiques : {len(df_n1):,}\n")

COLS_AFFICH = [
    "nmfinessetab_stru", "raisonsociale_stru", "email_stru",
    "code_anomalie", "libelle_anomalie", "suggestion",
]
cols = [c for c in COLS_AFFICH if c in df_n1.columns]

for code_a, groupe in df_n1.groupby("code_anomalie"):
    print(f"\n{'─'*60}")
    print(f"  {code_a}  ({len(groupe):,} cas)")
    print(f"  {groupe['libelle_anomalie'].iloc[0]}")
    print(f"{'─'*60}")
    print(groupe[cols].head(5).to_string(index=False))

## 10. Signalements qualité (niveau 2)

In [ ]:
df_n2 = df[df["niveau_anomalie"] == 2].copy()
print(f"Total signalements qualité : {len(df_n2):,}\n")

for code_a, groupe in df_n2.groupby("code_anomalie"):
    print(f"\n{'─'*60}")
    print(f"  {code_a}  ({len(groupe):,} cas)")
    print(f"  {groupe['libelle_anomalie'].iloc[0]}")
    print(f"{'─'*60}")
    print(groupe[cols].head(5).to_string(index=False))

## 11. Emails grand public — top domaines

In [13]:
df_gp = df_n2[df_n2["code_anomalie"] == "EMAIL_GRAND_PUBLIC"]

if len(df_gp) > 0:
    top_dom = df_gp["domaine"].value_counts().head(20)
    print(f"Top 20 domaines grand public ({len(df_gp):,} cas total) :\n")
    for domaine, n in top_dom.items():
        barre = "█" * min(int(n / top_dom.max() * 30), 30)
        print(f"  @{domaine:<28} {n:>6,}  {barre}")
else:
    print("Aucun email grand public détecté.")

Top 20 domaines grand public (19,822 cas total) :

  @gmail.com                     7,833  ██████████████████████████████
  @orange.fr                     4,899  ██████████████████
  @wanadoo.fr                    4,115  ███████████████
  @hotmail.fr                      642  ██
  @yahoo.fr                        624  ██
  @free.fr                         317  █
  @hotmail.com                     306  █
  @outlook.fr                      243  
  @laposte.net                     229  
  @yahoo.com                       111  
  @sfr.fr                           96  
  @live.fr                          83  
  @outlook.com                      53  
  @club-internet.fr                 50  
  @neuf.fr                          47  
  @bbox.fr                          40  
  @icloud.com                       39  
  @protonmail.com                   19  
  @me.com                           17  
  @aliceadsl.fr                     13  


## 12. Synthèse finale

In [14]:
n_total    = len(df)
n_vides    = (df["code_anomalie"] == "EMAIL_VIDE").sum()
n_renseign = max(n_total - n_vides, 1)
n_critique = (df["niveau_anomalie"] == 1).sum()
n_qualite  = (df["niveau_anomalie"] == 2).sum()
n_valides  = (df["niveau_anomalie"] == 0).sum()

print("=" * 60)
print("SYNTHÈSE GLOBALE — SIGNALEMENT EMAIL EG FINESS")
print("=" * 60)
print(f"  EG actifs analysés          : {n_total:>8,}")
print(f"  Email absent (vide)         : {n_vides:>8,}  ({n_vides/n_total*100:.1f} %)")
print()
print(f"  ── Sur les {n_renseign:,} emails renseignés ──")
print(f"  [1] Anomalies critiques     : {n_critique:>8,}")
print(f"  [2] Signalements qualité    : {n_qualite:>8,}")
print(f"  [0] Emails valides          : {n_valides:>8,}")
print("=" * 60)

SYNTHÈSE GLOBALE — SIGNALEMENT EMAIL EG FINESS
  EG actifs analysés          :  104,805
  Email absent (vide)         :   33,235  (31.7 %)

  ── Sur les 71,570 emails renseignés ──
  [1] Anomalies critiques     :   33,337
  [2] Signalements qualité    :   40,723
  [0] Emails valides          :   30,745


## 13. Export Excel — Rapport de signalement

In [15]:
from openpyxl.styles import PatternFill, Font, Alignment

DOSSIER_SORTIE = Path("../../results/email")
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
FICHIER_SORTIE = DOSSIER_SORTIE / "signalement_emails_eg.xlsx"

COLS_EXPORT = [
    "idstructure_stru", "nmfinessetab_stru", "nmfinessej_stru",
    "raisonsociale_stru", "cdcommune_stru", "departement", "lbvoie_stru",
    "email_stru", "email_norm",
    "code_anomalie", "libelle_anomalie", "suggestion",
    "type_local", "type_domaine", "correspondance",
    "geo_local", "geo_concordant",
]
cols_dispo = [c for c in COLS_EXPORT if c in df.columns]

def style_entete(ws, couleur):
    for cell in ws[1]:
        cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
        cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 28

def remplir(ws, couleur):
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
            cell.font = Font(name="Arial", size=9)

def auto_width(ws, max_w=50):
    for col in ws.columns:
        w = max((len(str(c.value or "")) for c in col), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(w + 3, max_w)

df_vides     = df[df["code_anomalie"] == "EMAIL_VIDE"][cols_dispo]
df_critiques = df[(df["niveau_anomalie"] == 1) & (df["code_anomalie"] != "EMAIL_VIDE")][cols_dispo]
df_doublons  = df[df["code_anomalie"] == "DOUBLON"][cols_dispo]
df_grandpub  = df[df["code_anomalie"] == "EMAIL_GRAND_PUBLIC"][cols_dispo]
df_valides   = df[df["niveau_anomalie"] == 0][cols_dispo]

# --- ajout : vue croisée par type de partie locale, tous type_domaine confondus (propre/public/doublon) ---
df_nominatifs   = df[df["type_local"].isin(["NOMINATIF", "NOMINATIF_PARTIEL"])][cols_dispo]
df_indetermines = df[df["type_local"] == "INDETERMINE"][cols_dispo]

df_synth = pd.DataFrame([
    {"Catégorie": "EG actifs total",            "Nombre": n_total,           "Part": "100 %"},
    {"Catégorie": "Email absent (vide)",         "Nombre": n_vides,           "Part": f"{n_vides/n_total*100:.1f} %"},
    {"Catégorie": "—",                           "Nombre": "",                "Part": ""},
    {"Catégorie": "[1] Anomalies critiques",     "Nombre": len(df_critiques), "Part": f"{len(df_critiques)/n_renseign*100:.1f} %"},
    {"Catégorie": "[2] Doublons",                "Nombre": len(df_doublons),  "Part": f"{len(df_doublons)/n_renseign*100:.1f} %"},
    {"Catégorie": "[2] Grand public",            "Nombre": len(df_grandpub),  "Part": f"{len(df_grandpub)/n_renseign*100:.1f} %"},
    {"Catégorie": "[0] Emails valides",          "Nombre": len(df_valides),   "Part": f"{len(df_valides)/n_renseign*100:.1f} %"},
    {"Catégorie": "—",                           "Nombre": "",                "Part": ""},
    {"Catégorie": "Vue croisée (hors total ci-dessus, quel que soit le domaine)", "Nombre": "", "Part": ""},
    {"Catégorie": "Nominatifs (NOMINATIF + PARTIEL)", "Nombre": len(df_nominatifs),   "Part": f"{len(df_nominatifs)/n_renseign*100:.1f} %"},
    {"Catégorie": "Indéterminés",                     "Nombre": len(df_indetermines), "Part": f"{len(df_indetermines)/n_renseign*100:.1f} %"},
])

with pd.ExcelWriter(FICHIER_SORTIE, engine="openpyxl") as writer:
    df_synth.to_excel(writer, sheet_name="Synthèse", index=False)
    style_entete(writer.sheets["Synthèse"], "1F3864")
    auto_width(writer.sheets["Synthèse"], max_w=40)

    for nom, donnees, ent, fond in [
        ("Emails_Vides",        df_vides,     "C62828", "F8CECC"),
        ("Anomalies_Critiques", df_critiques, "C62828", "F8CECC"),
        ("Doublons",            df_doublons,  "F57C00", "FFF2CC"),
        ("Grand_Public",        df_grandpub,  "F57C00", "FFF2CC"),
        ("Emails_Valides",      df_valides,   "2E7D32", "EBF5EB"),
        ("Nominatifs",          df_nominatifs,   "1565C0", "D6E4F0"), 
        ("Indetermines",        df_indetermines, "6A1B9A", "E8DAEF"),
    ]:
        if len(donnees) > 0:
            donnees.to_excel(writer, sheet_name=nom, index=False)
            style_entete(writer.sheets[nom], ent)
            remplir(writer.sheets[nom], fond)
            auto_width(writer.sheets[nom])

print(f"Export OK → {FICHIER_SORTIE.resolve()}")
print(f"\nFeuilles produites :")
print(f"  Emails_Vides         : {len(df_vides):,}")
print(f"  Anomalies_Critiques  : {len(df_critiques):,}")
print(f"  Doublons             : {len(df_doublons):,}")
print(f"  Grand_Public         : {len(df_grandpub):,}")
print(f"  Emails_Valides       : {len(df_valides):,}")
print(f"  Nominatifs           : {len(df_nominatifs):,}")
print(f"  Indetermines         : {len(df_indetermines):,}")

Export OK → /home/jovyan/work/signalement_data_contact/results/email/signalement_emails_eg.xlsx

Feuilles produites :
  Emails_Vides         : 33,235
  Anomalies_Critiques  : 102
  Doublons             : 20,901
  Grand_Public         : 19,822
  Emails_Valides       : 30,745
  Nominatifs           : 9,676
  Indetermines         : 24,451
